# SQL MVP — đánh giá Normal và GraphRAG

Notebook chạy **hai thí nghiệm độc lập**:

1. **Normal retrieval**: dev `mode`, `rrf_k`, trọng số dense/sparse; test bằng cấu hình Normal thắng trên dev.
2. **GraphRAG**: dev `community_algorithm`, `community_resolution`; test bằng cấu hình Graph thắng trên dev.

Mỗi luồng có bảng Dev và Test riêng. Bảng hiển thị là bảng báo cáo; CSV/JSON trong `data/eval/sql_mvp/benchmark` chứa kết quả đầy đủ theo cấu hình, knowledge và testcase. Test không tham gia chọn tham số.


In [1]:
from __future__ import annotations

import csv
import html
import json
import re
import sys
from pathlib import Path
from statistics import fmean

from IPython.display import HTML, display
import torch


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "branch_sql_MVP" / "settings.json").exists():
            return candidate
    raise RuntimeError("Không tìm thấy repo root")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.branch_sql_MVP.eval.graph import combine as combine_graph
from src.branch_sql_MVP.eval.graph import evaluate as evaluate_graph_structure
from src.branch_sql_MVP.eval.graph import load_cases
from src.branch_sql_MVP.offline.embed import _dense
from src.branch_sql_MVP.offline.graph import _communities, run as run_graph_extraction
from src.branch_sql_MVP.online.retrieval import _reranker, retrieve
from src.branch_sql_MVP.settings import Settings, load_settings

APP = load_settings()
OUTPUT_DIR = APP.path(APP.paths.eval) / "benchmark"
print(f"Repo: {ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Dense: {APP.embedding.dense_model}")
print(f"Reranker: {APP.embedding.rerank_model}")
print(f"Local model device: {APP.embedding.device} · CUDA available: {torch.cuda.is_available()}")
print(f"Graph LLM API: {APP.graph.provider}/{APP.graph.model}")
if APP.embedding.device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("settings yêu cầu CUDA nhưng torch không thấy GPU")


Repo: C:\Users\Khanh\Documents\vi-coze
Output: C:\Users\Khanh\Documents\vi-coze\data\eval\sql_mvp\benchmark
Dense: AITeamVN/Vietnamese_Embedding
Reranker: AITeamVN/Vietnamese_Reranker


Local model device: cuda · CUDA available: True
Graph LLM API: openai/gpt-5.6-luna


## 1. Tham số thí nghiệm

Toàn bộ giá trị cần thay đổi nằm trong cell này. Các model, collection, top-k mặc định và đường dẫn vẫn lấy từ `settings.json`.


In [2]:
RUN_DEV = True
RUN_TEST = True
RUN_GEMINI_GRAPH_TEST = True

NORMAL_DEV_KNOWLEDGE_IDS = [APP.index.knowledge_id]
NORMAL_TEST_KNOWLEDGE_IDS = list(APP.index.collections)
GRAPH_SOURCE_IDS = {
    "p1": ["mo_ta_bang_bds_new__docx", "text2sql_testcase__xlsx"],
    "p2": ["mo_ta_bang_bds_new__pdf", "text2sql_testcase__xlsx"],
}
GRAPH_DEV_KNOWLEDGE_IDS = [APP.index.knowledge_id]
GRAPH_TEST_KNOWLEDGE_IDS = list(GRAPH_SOURCE_IDS)
GRAPH_SOURCE_KINDS = {
    "mo_ta_bang_bds_new__docx": "docs",
    "mo_ta_bang_bds_new__pdf": "docs",
    "text2sql_testcase__xlsx": "sql",
}
GEMINI_GRAPH_PROVIDER = "google"
GEMINI_GRAPH_MODEL = "gemini-3.5-flash"
GEMINI_GRAPH_ARTIFACT_TAG = "gemini_3_5_flash"

GRAPH_SCORE_WEIGHTS = {"complete": 0.4, "connected": 0.3, "community": 0.3}

NORMAL_DEV_GRID = [
    {"mode": "semantic"},
    {"mode": "keyword"},
    {"mode": "hybrid", "rrf_k": 5, "semantic_weight": 0.1, "keyword_weight": 0.9},
    {"mode": "hybrid", "rrf_k": 5, "semantic_weight": 0.3, "keyword_weight": 0.7},
]

GRAPH_DEV_GRID = [
    {"community_algorithm": "louvain", "community_resolution": 0.5},
    {"community_algorithm": "louvain", "community_resolution": 1.0},
    {"community_algorithm": "louvain", "community_resolution": 1.5},
    {"community_algorithm": "louvain", "community_resolution": 2.0},
    {"community_algorithm": "greedy_modularity", "community_resolution": 1.0},
]


## 2. Helper chung và metric Normal

Gold là tập bảng trong `Relevant Chunks`. Metric đầy đủ gồm recall, precision, complete cho từng vế docs/SQL và hợp của hai vế.


In [3]:
SQL_TABLE_PATTERN = re.compile(
    r"\b(?:FROM|JOIN|UPDATE|INTO|MERGE\s+INTO)\s+([A-Za-z_][A-Za-z0-9_$#.]*)",
    re.IGNORECASE,
)
TABLE_FUNCTION_PATTERN = re.compile(r"\bTABLE\s*\(\s*([A-Za-z_][A-Za-z0-9_$#.]*)", re.IGNORECASE)
VI_TABLE_PATTERN = re.compile(r"\bBảng\s+(?:[*_`]+)?([A-Z][A-Z0-9_$#.\\-]*)")
GRAPH_TABLE_PATTERN = re.compile(r"^\s*table\s*:\s*([A-Za-z_][A-Za-z0-9_$#.]*)", re.IGNORECASE | re.MULTILINE)


def table_key(value: str) -> str:
    value = str(value or "").replace("\\_", "_").strip().upper()
    match = re.fullmatch(r"TABLE\s*\(\s*([^\)]+)\s*\)", value)
    if match:
        value = match.group(1)
    return re.sub(r"[^A-Z0-9_$#.]", "", value)


def tables_from_text(text: str) -> set[str]:
    text = str(text or "")
    values = (
        SQL_TABLE_PATTERN.findall(text)
        + TABLE_FUNCTION_PATTERN.findall(text)
        + VI_TABLE_PATTERN.findall(text)
        + GRAPH_TABLE_PATTERN.findall(text)
    )
    return {key for value in values if (key := table_key(value)) and key not in {"SELECT", "TABLE"}}


def tables_from_hit(hit: dict) -> set[str]:
    metadata = hit.get("metadata") or {}
    values: list[str] = []
    for field in ("table_name", "table", "relevant_table"):
        if metadata.get(field):
            values.append(str(metadata[field]))
    for field in ("tables", "relevant", "relevant_tables"):
        value = metadata.get(field)
        if isinstance(value, str):
            values.extend(value.split("|"))
        elif isinstance(value, (list, tuple, set)):
            values.extend(map(str, value))
    tables = {key for value in values if (key := table_key(value))}
    for field in ("text", "parent_text", "sql", "statement", "section"):
        tables.update(tables_from_text(metadata.get(field, "")))
    tables.update(tables_from_text(hit.get("text", "")))
    return tables


def gold_tables(case: dict) -> set[str]:
    return {key for value in case["relevant"] if (key := table_key(value))}


def recall(gold: set[str], predicted: set[str]) -> float:
    return len(gold & predicted) / len(gold) if gold else 1.0


def precision(gold: set[str], predicted: set[str]) -> float:
    return len(gold & predicted) / len(predicted) if predicted else 0.0


def f1_score(recall_value: float, precision_value: float) -> float:
    total = recall_value + precision_value
    return 2 * recall_value * precision_value / total if total else 0.0


def mean(values) -> float:
    values = list(values)
    return fmean(values) if values else 0.0


def normal_settings(overrides: dict) -> Settings:
    payload = {**APP.retrieval.model_dump(mode="json"), **overrides}
    retrieval_cfg = type(APP.retrieval).model_validate(payload)
    return APP.model_copy(update={"retrieval": retrieval_cfg})


def graph_settings(overrides: dict) -> Settings:
    payload = {**APP.graph.model_dump(mode="json"), **overrides}
    graph_cfg = type(APP.graph).model_validate(payload)
    return APP.model_copy(update={"graph": graph_cfg})


def normal_label(config: dict) -> str:
    if config["mode"] != "hybrid":
        return config["mode"]
    return f"hybrid_rrf{config['rrf_k']}_w{config['semantic_weight']:.1f}-{config['keyword_weight']:.1f}"


def graph_label(config: dict) -> str:
    return f"{config['community_algorithm']}_r{config['community_resolution']:g}"


def show_table(rows: list[dict], columns: list[str] | None = None) -> None:
    if not rows:
        display(HTML("<em>Không có kết quả.</em>"))
        return
    columns = columns or list(rows[0])
    head = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = []
    for row in rows:
        cells = []
        for column in columns:
            value = row.get(column, "")
            value = f"{value:.3f}" if isinstance(value, float) else value
            cells.append(f"<td>{html.escape(str(value))}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    style = ("<style>table.eval{border-collapse:collapse}table.eval th,table.eval td"
             "{border:1px solid #aaa;padding:6px 10px;text-align:right}"
             "table.eval th:first-child,table.eval td:first-child{text-align:left}</style>")
    display(HTML(style + f"<table class='eval'><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"))


def write_csv(name: str, rows: list[dict]) -> None:
    if not rows:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with (OUTPUT_DIR / name).open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_json(name: str, payload) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / name).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )


def evaluate_normal(cases: list[dict], split: str, knowledge_id: str, config: dict) -> dict:
    app = normal_settings(config)
    per_case = []
    for case in cases:
        gold = gold_tables(case)
        docs_hits = retrieve(case["query"], kind="docs", knowledge_id=knowledge_id, settings=app)
        sql_hits = retrieve(
            case["query"],
            kind="sql",
            knowledge_id=knowledge_id,
            exclude_section_ids={case["id"]} if split == "dev" else None,
            settings=app,
        )
        docs = set().union(*(tables_from_hit(hit) for hit in docs_hits)) if docs_hits else set()
        sql = set().union(*(tables_from_hit(hit) for hit in sql_hits)) if sql_hits else set()
        union = docs | sql
        per_case.append({
            "id": case["id"],
            "query": case["query"],
            "gold": " | ".join(sorted(gold)),
            "docs_predicted": " | ".join(sorted(docs)),
            "sql_predicted": " | ".join(sorted(sql)),
            "docs_recall": recall(gold, docs),
            "docs_precision": precision(gold, docs),
            "docs_complete": float(gold <= docs),
            "sql_recall": recall(gold, sql),
            "sql_precision": precision(gold, sql),
            "sql_complete": float(gold <= sql),
            "union_recall": recall(gold, union),
            "union_precision": precision(gold, union),
            "union_f1": f1_score(recall(gold, union), precision(gold, union)),
            "union_complete": float(gold <= union),
        })
    metrics = {
        "cases": len(per_case),
        f"docs@{app.retrieval.docs_top_k}": mean(row["docs_recall"] for row in per_case),
        "docs_precision": mean(row["docs_precision"] for row in per_case),
        "docs_complete": mean(row["docs_complete"] for row in per_case),
        f"sql@{app.retrieval.sql_top_k}": mean(row["sql_recall"] for row in per_case),
        "sql_precision": mean(row["sql_precision"] for row in per_case),
        "sql_complete": mean(row["sql_complete"] for row in per_case),
        "union_recall": mean(row["union_recall"] for row in per_case),
        "union_precision": mean(row["union_precision"] for row in per_case),
        "union_f1": mean(row["union_f1"] for row in per_case),
        "union_complete": mean(row["union_complete"] for row in per_case),
    }
    metrics["score"] = metrics["union_f1"]
    return {"metrics": metrics, "per_case": per_case}


## 3. Normal — DEV

Chạy toàn bộ cấu hình Normal trên sheet `dev` bằng leave-one-out SQL (không được truy hồi chính case đang chấm), xếp theo macro `union_f1` và khóa cấu hình đứng đầu.


In [4]:
NORMAL_DEV_SUMMARY = []
NORMAL_DEV_BY_KNOWLEDGE = []
NORMAL_DEV_PER_CASE = []
NORMAL_LOCKED = None

if RUN_DEV:
    dev_cases = load_cases("dev", settings=APP)
    for overrides in NORMAL_DEV_GRID:
        config = normal_settings(overrides).retrieval.model_dump(mode="json")
        label = normal_label(config)
        rows = []
        for knowledge_id in NORMAL_DEV_KNOWLEDGE_IDS:
            result = evaluate_normal(dev_cases, "dev", knowledge_id, config)
            row = {
                "config": label,
                "knowledge_id": knowledge_id,
                "mode": config["mode"],
                "rrf_k": config["rrf_k"] if config["mode"] == "hybrid" else "—",
                "w_dense/sparse": (
                    f"{config['semantic_weight']:.1f}/{config['keyword_weight']:.1f}"
                    if config["mode"] == "hybrid" else "—"
                ),
                **result["metrics"],
            }
            rows.append(row)
            NORMAL_DEV_BY_KNOWLEDGE.append(row)
            NORMAL_DEV_PER_CASE.extend({"config": label, "knowledge_id": knowledge_id, **item} for item in result["per_case"])
        metric_names = list(result["metrics"])
        NORMAL_DEV_SUMMARY.append({
            "config": label,
            "mode": rows[0]["mode"],
            "rrf_k": rows[0]["rrf_k"],
            "w_dense/sparse": rows[0]["w_dense/sparse"],
            **{name: mean(row[name] for row in rows) for name in metric_names},
            "_config": config,
        })
    NORMAL_DEV_SUMMARY.sort(key=lambda row: (row["score"], row["union_complete"], row["union_recall"]), reverse=True)
    NORMAL_LOCKED = dict(NORMAL_DEV_SUMMARY[0]["_config"])
    write_json("normal_locked.json", NORMAL_LOCKED)
    write_csv("normal_dev_summary.csv", [{key: value for key, value in row.items() if not key.startswith("_")} for row in NORMAL_DEV_SUMMARY])
    write_csv("normal_dev_by_knowledge.csv", NORMAL_DEV_BY_KNOWLEDGE)
    write_csv("normal_dev_per_case.csv", NORMAL_DEV_PER_CASE)
    show_table(
        [{key: value for key, value in row.items() if not key.startswith("_")} for row in NORMAL_DEV_SUMMARY],
        ["config", f"docs@{APP.retrieval.docs_top_k}", f"sql@{APP.retrieval.sql_top_k}", "union_recall", "union_precision", "union_f1"],
    )
    print(f"Normal locked: {normal_label(NORMAL_LOCKED)}")
    dense_device = str(_dense(APP.embedding.dense_model, APP.embedding.device).device)
    reranker_device = str(next(_reranker(APP.embedding.rerank_model, APP.embedding.device).model.parameters()).device)
    print(f"Dense runtime device: {dense_device}")
    print(f"Reranker runtime device: {reranker_device}")
    if APP.embedding.device == "cuda" and (not dense_device.startswith("cuda") or not reranker_device.startswith("cuda")):
        raise RuntimeError("dense/reranker không chạy trên CUDA")
else:
    NORMAL_LOCKED = json.loads((OUTPUT_DIR / "normal_locked.json").read_text(encoding="utf-8"))


C:\Users\Khanh\Documents\vi-coze\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:  57%|█████▋    | 221/391 [00:00<00:00, 1974.40it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2046.09it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  39%|███▉      | 155/393 [00:00<00:00, 1382.85it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2263.72it/s]

config,docs@5,sql@3,union_recall,union_precision,union_f1
hybrid_rrf5_w0.1-0.9,0.467,0.949,0.960,0.440,0.578
hybrid_rrf5_w0.3-0.7,0.477,0.914,0.970,0.436,0.574
keyword,0.486,0.949,0.960,0.423,0.559
semantic,0.396,0.929,0.960,0.370,0.514


Normal locked: hybrid_rrf5_w0.1-0.9
Dense runtime device: cuda:0
Reranker runtime device: cuda:0


## 4. Normal — TEST thực tế

Chỉ chạy cấu hình Normal đã khóa trên sheet `test`; không xếp hạng hay chọn lại tham số.


In [5]:
NORMAL_TEST = []
NORMAL_TEST_PER_CASE = []

if RUN_TEST:
    test_cases = load_cases("test", settings=APP)
    for knowledge_id in NORMAL_TEST_KNOWLEDGE_IDS:
        result = evaluate_normal(test_cases, "test", knowledge_id, NORMAL_LOCKED)
        NORMAL_TEST.append({
            "config": normal_label(NORMAL_LOCKED),
            "knowledge_id": knowledge_id,
            **result["metrics"],
        })
        NORMAL_TEST_PER_CASE.extend({"knowledge_id": knowledge_id, **item} for item in result["per_case"])
    write_csv("normal_test.csv", NORMAL_TEST)
    write_csv("normal_test_per_case.csv", NORMAL_TEST_PER_CASE)
    show_table(NORMAL_TEST, ["config", "knowledge_id", f"docs@{APP.retrieval.docs_top_k}", f"sql@{APP.retrieval.sql_top_k}", "union_recall", "union_precision", "union_f1"] )


config,knowledge_id,docs@5,sql@3,union_recall,union_precision,union_f1
hybrid_rrf5_w0.1-0.9,p1,0.405,0.970,0.988,0.505,0.653
hybrid_rrf5_w0.1-0.9,p2,0.506,0.970,0.988,0.415,0.561


## 5. GraphRAG — metric và DEV

Graph được LLM extract một lần ở offline. Dev chỉ dùng graph tài liệu để tránh graph SQL chứa chính case dev; test dùng graph tài liệu + SQL sample đã khóa. `connected_components` không tham gia tuning vì nó biến `same_community` thành bản sao của `connected`, tạo lợi thế metric mặc định.

`graph_score = 0.4 × complete + 0.3 × connected + 0.3 × same_community` theo mặc định; trọng số chỉnh ở cell tham số.


In [6]:
def evaluate_graph(
    split: str,
    knowledge_id: str,
    config: dict,
    *,
    artifact_tag: str | None = None,
    output_tag: str | None = None,
) -> dict:
    app = graph_settings(config)
    source_ids = GRAPH_SOURCE_IDS[knowledge_id][:1] if split == "dev" else GRAPH_SOURCE_IDS[knowledge_id]
    artifact = combine_graph(source_ids, artifact_tag=artifact_tag, settings=app)
    nx_graph, communities = _communities(artifact["nodes"], artifact["edges"], app.graph)
    artifact["communities"] = communities
    artifact["stats"].update({
        "communities": len(communities),
        "isolated_nodes": len(list(__import__("networkx").isolates(nx_graph))),
    })
    result = evaluate_graph_structure(
        artifact["doc_id"], knowledge_id=knowledge_id, split=split, artifact=artifact,
        output_tag=output_tag, settings=app
    )
    metrics = dict(result["metrics"])
    stats = result.get("graph_stats", {})
    graph_score = (
        GRAPH_SCORE_WEIGHTS["complete"] * metrics["complete_case_rate"]
        + GRAPH_SCORE_WEIGHTS["connected"] * metrics["connected_case_rate"]
        + GRAPH_SCORE_WEIGHTS["community"] * metrics["same_community_rate"]
    )
    summary = {
        "cases": metrics["cases"],
        "nodes": stats.get("nodes", 0),
        "edges": stats.get("edges", 0),
        "communities": stats.get("communities", 0),
        "isolated_nodes": stats.get("isolated_nodes", 0),
        "table_coverage": metrics["table_coverage"],
        "macro_table_recall": metrics["macro_table_recall"],
        "complete_case_rate": metrics["complete_case_rate"],
        "connected_case_rate": metrics["connected_case_rate"],
        "same_community_rate": metrics["same_community_rate"],
        "mean_shortest_path": metrics["mean_shortest_path"],
        "graph_score": graph_score,
    }
    per_case = [{
        **item,
        "relevant": " | ".join(item["relevant"]),
        "matched": " | ".join(item["matched"]),
        "missing": " | ".join(item["missing"]),
    } for item in result["per_case"]]
    return {"summary": summary, "per_case": per_case}


GRAPH_DEV_SUMMARY = []
GRAPH_DEV_BY_KNOWLEDGE = []
GRAPH_DEV_PER_CASE = []
GRAPH_LOCKED = None

if RUN_DEV:
    for overrides in GRAPH_DEV_GRID:
        config = graph_settings(overrides).graph.model_dump(mode="json")
        label = graph_label(config)
        rows = []
        for knowledge_id in GRAPH_DEV_KNOWLEDGE_IDS:
            result = evaluate_graph("dev", knowledge_id, config)
            row = {
                "config": label,
                "knowledge_id": knowledge_id,
                "algorithm": config["community_algorithm"],
                "resolution": config["community_resolution"],
                **result["summary"],
            }
            rows.append(row)
            GRAPH_DEV_BY_KNOWLEDGE.append(row)
            GRAPH_DEV_PER_CASE.extend({"config": label, "knowledge_id": knowledge_id, **item} for item in result["per_case"])
        metric_names = list(result["summary"])
        GRAPH_DEV_SUMMARY.append({
            "config": label,
            "algorithm": config["community_algorithm"],
            "resolution": config["community_resolution"],
            **{name: mean(row[name] for row in rows) for name in metric_names},
            "_config": config,
        })
    GRAPH_DEV_SUMMARY.sort(key=lambda row: (row["graph_score"], row["same_community_rate"], row["connected_case_rate"]), reverse=True)
    GRAPH_LOCKED = dict(GRAPH_DEV_SUMMARY[0]["_config"])
    write_json("graph_locked.json", GRAPH_LOCKED)
    write_csv("graph_dev_summary.csv", [{key: value for key, value in row.items() if not key.startswith("_")} for row in GRAPH_DEV_SUMMARY])
    write_csv("graph_dev_by_knowledge.csv", GRAPH_DEV_BY_KNOWLEDGE)
    write_csv("graph_dev_per_case.csv", GRAPH_DEV_PER_CASE)
    show_table(
        [{key: value for key, value in row.items() if not key.startswith("_")} for row in GRAPH_DEV_SUMMARY],
        ["config", "communities", "isolated_nodes", "connected_case_rate", "same_community_rate", "graph_score"],
    )
    print(f"Graph locked: {graph_label(GRAPH_LOCKED)}")
else:
    GRAPH_LOCKED = json.loads((OUTPUT_DIR / "graph_locked.json").read_text(encoding="utf-8"))


config,communities,isolated_nodes,connected_case_rate,same_community_rate,graph_score
greedy_modularity_r1,60.000,51.000,1.000,0.394,0.818
louvain_r0.5,57.000,51.000,1.000,0.212,0.764
louvain_r1,59.000,51.000,1.000,0.212,0.764
louvain_r1.5,62.000,51.000,1.000,0.212,0.764
louvain_r2,64.000,51.000,1.000,0.212,0.764


Graph locked: greedy_modularity_r1


## 6. GraphRAG — TEST thực tế

Gộp đúng Graph `docs + sql` đã extract, áp dụng cấu hình community đã khóa và đo trên sheet `test`. Không chọn lại tham số theo kết quả test.


In [7]:
GRAPH_TEST = []
GRAPH_TEST_PER_CASE = []

if RUN_TEST:
    for knowledge_id in GRAPH_TEST_KNOWLEDGE_IDS:
        result = evaluate_graph("test", knowledge_id, GRAPH_LOCKED)
        GRAPH_TEST.append({
            "provider": APP.graph.provider,
            "model": APP.graph.model,
            "config": graph_label(GRAPH_LOCKED),
            "knowledge_id": knowledge_id,
            "algorithm": GRAPH_LOCKED["community_algorithm"],
            "resolution": GRAPH_LOCKED["community_resolution"],
            **result["summary"],
        })
        GRAPH_TEST_PER_CASE.extend({"knowledge_id": knowledge_id, **item} for item in result["per_case"])
    write_csv("graph_test.csv", GRAPH_TEST)
    write_csv("graph_test_per_case.csv", GRAPH_TEST_PER_CASE)
    show_table(GRAPH_TEST, ["model", "knowledge_id", "nodes", "edges", "isolated_nodes", "connected_case_rate", "same_community_rate", "graph_score"] )


model,knowledge_id,nodes,edges,isolated_nodes,connected_case_rate,same_community_rate,graph_score
gpt-5.6-luna,p1,392,562,47,0.786,0.071,0.657
gpt-5.6-luna,p2,401,553,41,0.786,0.071,0.657


## 7. GraphRAG — so sánh Gemini 3.5 Flash

Extract lại cùng các nguồn bằng `gemini-3.5-flash`, lưu artifact có tag riêng và chấm bằng đúng cấu hình community đã khóa. Artifact và CSV của GPT hiện tại không bị ghi đè.


In [8]:
GEMINI_GRAPH_TEST = []
GEMINI_GRAPH_TEST_PER_CASE = []
GRAPH_MODEL_TEST = []

if RUN_GEMINI_GRAPH_TEST:
    gemini_config = {
        **GRAPH_LOCKED,
        "provider": GEMINI_GRAPH_PROVIDER,
        "model": GEMINI_GRAPH_MODEL,
    }
    gemini_app = graph_settings(gemini_config)
    source_ids = list(dict.fromkeys(
        doc_id for ids in GRAPH_SOURCE_IDS.values() for doc_id in ids
    ))
    for doc_id in source_ids:
        result = run_graph_extraction(
            doc_id,
            kind=GRAPH_SOURCE_KINDS[doc_id],
            provider=GEMINI_GRAPH_PROVIDER,
            model=GEMINI_GRAPH_MODEL,
            artifact_tag=GEMINI_GRAPH_ARTIFACT_TAG,
            settings=gemini_app,
        )
        print(
            f"Gemini graph {doc_id}: {result['nodes']} nodes · "
            f"{result['edges']} edges · {result['failed_chunks']} failed chunks"
        )

    for knowledge_id in GRAPH_TEST_KNOWLEDGE_IDS:
        result = evaluate_graph(
            "test",
            knowledge_id,
            gemini_config,
            artifact_tag=GEMINI_GRAPH_ARTIFACT_TAG,
            output_tag=GEMINI_GRAPH_ARTIFACT_TAG,
        )
        GEMINI_GRAPH_TEST.append({
            "provider": GEMINI_GRAPH_PROVIDER,
            "model": GEMINI_GRAPH_MODEL,
            "config": graph_label(gemini_config),
            "knowledge_id": knowledge_id,
            **result["summary"],
        })
        GEMINI_GRAPH_TEST_PER_CASE.extend(
            {"model": GEMINI_GRAPH_MODEL, "knowledge_id": knowledge_id, **item}
            for item in result["per_case"]
        )

    GRAPH_MODEL_TEST = [*GRAPH_TEST, *GEMINI_GRAPH_TEST]
    write_csv("graph_gemini_test.csv", GEMINI_GRAPH_TEST)
    write_csv("graph_gemini_test_per_case.csv", GEMINI_GRAPH_TEST_PER_CASE)
    write_csv("graph_model_test.csv", GRAPH_MODEL_TEST)
    show_table(
        GRAPH_MODEL_TEST,
        ["model", "knowledge_id", "nodes", "edges", "isolated_nodes",
         "connected_case_rate", "same_community_rate", "graph_score"],
    )


## File kết quả

### Normal

- `normal_dev_summary.csv`: toàn bộ cấu hình Dev — bảng dùng chọn tham số.
- `normal_dev_by_knowledge.csv`, `normal_dev_per_case.csv`: kết quả chi tiết Dev.
- `normal_test.csv`, `normal_test_per_case.csv`: kết quả Test thực tế.
- `normal_locked.json`: cấu hình Normal thắng Dev.

### GraphRAG

- `graph_dev_summary.csv`: toàn bộ cấu hình Graph Dev.
- `graph_dev_by_knowledge.csv`, `graph_dev_per_case.csv`: kết quả chi tiết Graph Dev.
- `graph_test.csv`, `graph_test_per_case.csv`: kết quả Graph Test thực tế.
- `graph_gemini_test.csv`, `graph_gemini_test_per_case.csv`: kết quả riêng Gemini 3.5 Flash.
- `graph_model_test.csv`: bảng gọn so sánh GPT và Gemini.
- `graph_locked.json`: cấu hình Graph thắng Dev.
